<a href="https://colab.research.google.com/github/oihika/AI-Enabled-Predictive-Energy-Management-for-Smart-Sustainable-Communities/blob/main/DASPI_REVIEWER_RECTIFIED_COMPLETE_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json  # required by NASA POWER request/response archival
# ================================================================
# DASPI: Duration-Adaptive Storage Portfolio Intelligence
# ================================================================
# Research prototype (reviewer-revised; scenario results are not validated regional estimates) for:
# "Duration-Adaptive Storage Portfolio Intelligence (DASPI)
#  for High-Renewable-Penetration Power Systems"
#
# Author: Oihika Arpit
#
# IMPORTANT:
# This is a reproducible RESEARCH PROTOTYPE.
# Do not present generated results as final empirical findings
# until the assumptions, datasets and technology parameters
# are replaced/validated with cited sources.
#
# Main architecture:
#
# Coordinates
#     ↓
# NASA POWER renewable resource data
#     ↓
# Solar + Wind + Hydro + Demand
#     ↓
# Renewable-deficit signature
#     ↓
# Duration/severity analysis
#     ↓
# DMI
#     ↓
# Multi-technology optimisation
#     ↓
# Li-ion / Flow / PHS / H2 / Thermal
#     ↓
# DMI + RASV + TSCR
#     ↓
# Comparative + sensitivity analysis
#     ↓
# Publication figures + CSV results
# ================================================================


# ================================================================
# 0. INSTALL / IMPORT
# ================================================================

import sys
import subprocess
import importlib

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "requests": "requests"
}

for package, pip_name in required_packages.items():
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name]
        )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import math
import os
import warnings

from scipy.optimize import milp, LinearConstraint, Bounds

warnings.filterwarnings("default")

# ================================================================
# REVIEWER REVISION: REPRODUCIBILITY AND EXPLICIT ASSUMPTIONS
# ================================================================
from datetime import datetime, timezone
import hashlib

OUTPUT_DIR = os.environ.get("DASPI_OUTPUT_DIR", "/content/DASPI_Research_Outputs")
RAW_DIR = os.path.join(OUTPUT_DIR, "data", "raw")
PROCESSED_DIR = os.path.join(OUTPUT_DIR, "data", "processed")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
for _folder in (RAW_DIR, PROCESSED_DIR, RESULTS_DIR):
    os.makedirs(_folder, exist_ok=True)

MODEL_LIFETIME_YEARS = 20.0
MAX_STORAGE_POWER_MW = 5000.0
MAX_STORAGE_ENERGY_MWH = 50000.0

# Explicit assumed resource-to-power conversion parameters. These are
# simplified screening assumptions, not a plant-specific PV/turbine model.
SOLAR_INPUT_UNIT_ASSUMPTION = "Require NASA POWER response metadata to report irradiance units; conversion uses the verified response unit and is archived"
PV_SYSTEM_DERATE = 0.80
PV_TEMPERATURE_COEFFICIENT_PER_C = -0.004
PV_REFERENCE_TEMPERATURE_C = 25.0
WIND_MEASUREMENT_HEIGHT_M = 10.0
WIND_HUB_HEIGHT_M = 100.0
WIND_SHEAR_EXPONENT = 0.14
WIND_TURBINE_MODEL = "generic normalized curve; no manufacturer-specific curve"

# Optional user-provided empirical inputs. If absent, constructed scenarios
# are used and must not be described as observed regional data.
DEMAND_CSV_PATH = os.environ.get("DASPI_DEMAND_CSV", "")
HYDRO_CSV_PATH = os.environ.get("DASPI_HYDRO_CSV", "")


print("DASPI environment ready.")


# ================================================================
# 1. USER CONFIGURATION
# ================================================================
#
# CHANGE THESE VALUES FOR YOUR CASE STUDY.
#
# Example:
# Jabalpur:     23.1815, 79.9864
# Bhopal:       23.2599, 77.4126
# Indore:       22.7196, 75.8577
#
# You can enter ANY location within valid geographic bounds.

LATITUDE = 23.1815
LONGITUDE = 79.9864

LOCATION_NAME = "Jabalpur, Madhya Pradesh, India"

# Coordinate constraint:
# Set this to True if you want the optimisation study restricted
# to a predefined geographical study region.

USE_COORDINATE_CONSTRAINT = True

# Example Central India boundary
LAT_MIN = 20.0
LAT_MAX = 25.5
LON_MIN = 74.0
LON_MAX = 82.5


# ================================================================
# TIME / DATA CONFIGURATION
# ================================================================

START_DATE = "2025-01-01"
END_DATE   = "2025-12-31"

# 4-hour resolution is recommended for Colab prototype optimisation.
#
# For final research:
# RESAMPLE_HOURS = 1
#
# will provide hourly modelling but will require substantially
# more computational resources.

RESAMPLE_HOURS = 4

TIME_STEP = RESAMPLE_HOURS


# ================================================================
# RENEWABLE CONFIGURATION
# ================================================================

TARGET_RENEWABLE_SHARE = 0.70

# Solar / wind share of renewable capacity proxy
SOLAR_SHARE = 0.60
WIND_SHARE = 0.40

# Hydro is treated as an additional dispatchable renewable source.
HYDRO_CAPACITY_MW = 100.0

# Maximum renewable installed capacity allowed
MAX_SOLAR_MW = 5000.0
MAX_WIND_MW = 5000.0


# ================================================================
# DEFICIT DETECTION PARAMETERS
# ================================================================

# Deficit threshold:
# Positive residual demand means renewable generation
# is insufficient to meet demand.

DEFICIT_THRESHOLD_MW = 0.0

# Minimum duration of an event
MIN_DEFICIT_DURATION_HOURS = 4


# ================================================================
# STORAGE TECHNOLOGIES
# ================================================================
#
# Values below are MODEL ASSUMPTIONS for the prototype.
# Replace them with properly sourced values before publication.
#
# power_capex: USD/kW
# energy_capex: USD/kWh
# variable_cost: assumed USD/kWh
# eta_ch: charging efficiency
# eta_dis: discharging efficiency
# self_discharge: hourly fractional loss
# min_duration / max_duration: technology operating range
#
# Duration is NOT imposed as the final answer.
# The optimisation chooses E/P inside the technology's
# feasible operating envelope.

STORAGE = {

    "Li-ion": {
        "power_capex": 300,
        "energy_capex": 180,
        "eta_ch": 0.95,
        "eta_dis": 0.95,
        "self_discharge": 0.0001,
        "min_duration": 1,
        "max_duration": 8,
        "variable_cost": 0.002
    },

    "Flow Battery": {
        "power_capex": 500,
        "energy_capex": 100,
        "eta_ch": 0.88,
        "eta_dis": 0.88,
        "self_discharge": 0.00005,
        "min_duration": 4,
        "max_duration": 24,
        "variable_cost": 0.003
    },

    "Pumped Hydro": {
        "power_capex": 1200,
        "energy_capex": 40,
        "eta_ch": 0.85,
        "eta_dis": 0.90,
        "self_discharge": 0.00001,
        "min_duration": 6,
        "max_duration": 100,
        "variable_cost": 0.001
    },

    "Hydrogen": {
        "power_capex": 900,
        "energy_capex": 15,
        "eta_ch": 0.65,
        "eta_dis": 0.55,
        "self_discharge": 0.000001,
        "min_duration": 12,
        "max_duration": 1000,
        "variable_cost": 0.010
    },

    "Thermal Storage": {
        "power_capex": 400,
        "energy_capex": 50,
        "eta_ch": 0.80,
        "eta_dis": 0.80,
        "self_discharge": 0.001,
        "min_duration": 4,
        "max_duration": 72,
        "variable_cost": 0.004
    }
}

TECHNOLOGIES = list(STORAGE.keys())
N_TECH = len(TECHNOLOGIES)


# ================================================================
# 2. COORDINATE VALIDATION
# ================================================================

def validate_coordinates(lat, lon):

    if not (-90 <= lat <= 90):
        raise ValueError("Latitude must be between -90 and +90 degrees.")

    if not (-180 <= lon <= 180):
        raise ValueError("Longitude must be between -180 and +180 degrees.")

    if USE_COORDINATE_CONSTRAINT:

        inside = (
            LAT_MIN <= lat <= LAT_MAX
            and
            LON_MIN <= lon <= LON_MAX
        )

        if not inside:
            raise ValueError(
                f"""
COORDINATE CONSTRAINT VIOLATION

Selected coordinate:
Latitude  = {lat}
Longitude = {lon}

Allowed study region:
Latitude  = {LAT_MIN} to {LAT_MAX}
Longitude = {LON_MIN} to {LON_MAX}

Change the coordinates or disable USE_COORDINATE_CONSTRAINT.
"""
            )

    print("\nCoordinate validation successful.")
    print(f"Latitude  : {lat:.6f}")
    print(f"Longitude : {lon:.6f}")
    print(f"Location  : {LOCATION_NAME}")

    return True


validate_coordinates(LATITUDE, LONGITUDE)


# ================================================================
# 3. NASA POWER DATA DOWNLOAD
# ================================================================

def get_nasa_power_data(lat, lon, start_date, end_date):
    """Fetch NASA POWER hourly data and archive the exact request and raw response."""
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
    params = {
        "parameters": "ALLSKY_SFC_SW_DWN,WS10M,T2M",
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": start_date.replace("-", ""),
        "end": end_date.replace("-", ""),
        "format": "JSON",
        "time-standard": "UTC"
    }
    retrieved_utc = datetime.now(timezone.utc).isoformat()
    print("\nDownloading NASA POWER renewable-resource data...")
    response = requests.get(url, params=params, timeout=90)
    response.raise_for_status()
    payload = response.json()
    # Preserve raw response immediately, even if a later unit-validation check fails.
    with open(os.path.join(RAW_DIR, "nasa_power_hourly_raw.json"), "w", encoding="utf-8") as f:
        json.dump(payload, f)

    with open(os.path.join(RAW_DIR, "nasa_power_request_metadata.json"), "w", encoding="utf-8") as f:
        json.dump({
            "requested_url": url, "resolved_url": response.url,
            "parameters": params, "retrieved_at_utc": retrieved_utc,
            "http_status": response.status_code,
            "response_sha256": hashlib.sha256(response.content).hexdigest(),
            "validation_status": "pending unit validation"
        }, f, indent=2)
    parameter_metadata = payload.get("parameters", {})
    response_units = {
        key: parameter_metadata.get(key, {}).get("units")
        for key in ("ALLSKY_SFC_SW_DWN", "WS10M", "T2M")
    }
    solar_unit = str(response_units.get("ALLSKY_SFC_SW_DWN", "")).lower().replace(" ", "")
    if not any(u in solar_unit for u in ["w/m", "wh/m", "kwh/m", "watt"]):
        raise ValueError(
            "NASA POWER solar units were not verified as irradiance. "
            f"Response units={response_units}. Do not run PV conversion until units are resolved."
          )
    wind_unit = str(response_units.get("WS10M", "")).lower().replace(" ", "")
    if not ("m/s" in wind_unit or "m.s-1" in wind_unit or "msec-1" in wind_unit):
        raise ValueError(f"NASA POWER wind units were not verified as m/s: {response_units}")

    request_record = {
        "requested_url": url,
        "resolved_url": response.url,
        "parameters": params,
        "retrieved_at_utc": retrieved_utc,
        "http_status": response.status_code,
        "response_sha256": hashlib.sha256(response.content).hexdigest(),
        "response_header": payload.get("header", {}),
        "response_parameter_metadata": parameter_metadata,
        "verified_response_units": response_units,
        "solar_unit_assumption": SOLAR_INPUT_UNIT_ASSUMPTION,
        "wind_measurement_height_m": WIND_MEASUREMENT_HEIGHT_M,
        "wind_hub_height_m": WIND_HUB_HEIGHT_M,
        "wind_shear_exponent_assumption": WIND_SHEAR_EXPONENT,
        "pv_system_derate_assumption": PV_SYSTEM_DERATE,
        "pv_temperature_coefficient_per_C_assumption": PV_TEMPERATURE_COEFFICIENT_PER_C
    }
    with open(os.path.join(RAW_DIR, "nasa_power_request_metadata.json"), "w", encoding="utf-8") as f:
        json.dump(request_record, f, indent=2)
    with open(os.path.join(RAW_DIR, "nasa_power_hourly_raw.json"), "w", encoding="utf-8") as f:
        json.dump(payload, f)

    properties = payload["properties"]["parameter"]
    solar = properties["ALLSKY_SFC_SW_DWN"]
    wind = properties["WS10M"]
    temp = properties["T2M"]
    rows = []
    for timestamp in solar.keys():
        rows.append({
            "datetime": pd.to_datetime(timestamp, format="%Y%m%d%H", utc=True),
            "solar_resource": solar[timestamp],
            "wind_speed_10m": wind.get(timestamp, np.nan),
            "temperature_C": temp.get(timestamp, np.nan)
        })
    out = pd.DataFrame(rows).sort_values("datetime").set_index("datetime")
    out = out.replace([-999, -999.0], np.nan)
    missing_before = out.isna().sum().to_dict()
    # Interpolate at most two consecutive missing hourly values; raw JSON is preserved.
    out = out.interpolate(limit=2, limit_area="inside")
    missing_after = out.isna().sum().to_dict()
    out = out.dropna()
    expected = (pd.Timestamp(end_date, tz="UTC") - pd.Timestamp(start_date, tz="UTC")).days * 24 + 24
    print(f"Valid hourly observations: {len(out):,}; expected approximately {expected:,}.")
    print("Missing counts before limited interpolation:", missing_before)
    print("Missing counts after limited interpolation:", missing_after)
    if len(out) == 0:
        raise ValueError("NASA POWER returned no valid rows; inspect raw response and units.")
    out.to_csv(os.path.join(PROCESSED_DIR, "nasa_power_hourly_processed.csv"))
    return out


resource = get_nasa_power_data(
    LATITUDE,
    LONGITUDE,
    START_DATE,
    END_DATE
)

# ================================================================
# 7. RESAMPLE DATA (Moved early to initialize resampled df)
# ================================================================

df = resource.resample(
    f"{RESAMPLE_HOURS}h"
).mean()

df = df.dropna()

# ================================================================
# 4. RENEWABLE RESOURCE CHECK (Performed on resampled df)
# ================================================================

print("\n===============================================")
print("LOCATION RESOURCE DIAGNOSTIC")
print("===============================================")

print(
    f"Mean solar resource indicator : "
    f"{df['solar_resource'].mean():.3f}"
)

print(
    f"Mean wind speed               : "
    f"{df['wind_speed_10m'].mean():.3f} m/s"
)

print(
    f"Maximum wind speed            : "
    f"{df['wind_speed_10m'].max():.3f} m/s"
)

print(
    f"Mean temperature              : "
    f"{df['temperature_C'].mean():.2f} °C"
)


# ================================================================
# 5. WIND POWER CURVE (Operating directly on resampled df)
# ================================================================

def wind_power_curve(v_hub_m_s):
    """Generic normalized curve: cut-in 3, rated 12, cut-out 25 m/s."""
    v = np.asarray(v_hub_m_s, dtype=float)
    power = np.zeros_like(v, dtype=float)
    mask1 = (v >= 3.0) & (v < 12.0)
    power[mask1] = (v[mask1]**3 - 3.0**3) / (12.0**3 - 3.0**3)
    mask2 = (v >= 12.0) & (v <= 25.0)
    power[mask2] = 1.0
    return np.clip(power, 0.0, 1.0)

# Explicit power-law adjustment from 10 m to assumed 100 m hub height.
# The shear exponent is an assumption; replace with site-specific evidence.
df["wind_speed_hub_m_s"] = (
    df["wind_speed_10m"]
    * (WIND_HUB_HEIGHT_M / WIND_MEASUREMENT_HEIGHT_M) ** WIND_SHEAR_EXPONENT
)
df["wind_speed"] = df["wind_speed_10m"]  # compatibility alias for reporting
df["wind_cf"] = wind_power_curve(df["wind_speed_hub_m_s"].values)


# ================================================================
# 6. SOLAR NORMALISATION (Operating directly on resampled df)
# ================================================================

# Simplified resource-to-PV conversion with explicit, auditable assumptions.
# Verify the NASA POWER parameter units from the archived raw response metadata.
# This is not a plant-specific PV model and must not be described as PVsyst.
solar_irradiance_w_m2 = df["solar_resource"].clip(lower=0)
temp_factor = 1.0 + PV_TEMPERATURE_COEFFICIENT_PER_C * (
    df["temperature_C"] - PV_REFERENCE_TEMPERATURE_C
)
temp_factor = temp_factor.clip(lower=0.0, upper=1.2)
df["solar_cf"] = (
    (solar_irradiance_w_m2 / 1000.0) * PV_SYSTEM_DERATE * temp_factor
).clip(lower=0.0, upper=1.0)
if df["solar_cf"].max() <= 0:
    raise ValueError("Solar resource data has no usable values; verify NASA POWER units.")


# ================================================================
# 8. REPRESENTATIVE ELECTRICITY DEMAND
# ================================================================
#
# IMPORTANT:
# This is a transparent modelling profile, NOT measured electricity
# demand. Replace it with actual Central India demand data for
# publication.
#
# The profile contains:
# - daily cycle
# - evening peak
# - seasonal variation
# - weekday/weekend effect

def create_demand_profile(index):

    hour = index.hour.values
    dayofyear = index.dayofyear.values
    weekday = index.dayofweek.values

    daily_component = (
        0.65
        + 0.15 * np.sin(
            2 * np.pi * (hour - 7) / 24
        )
        + 0.20 * np.exp(
            -((hour - 20) / 3.0)**2
        )
    )

    seasonal_component = (
        1.0
        + 0.10 * np.cos(
            2 * np.pi * (dayofyear - 200) / 365
        )
    )

    weekend_component = np.where(
        weekday >= 5,
        0.92,
        1.0
    )

    demand = (
        daily_component
        * seasonal_component
        * weekend_component
    )

    # Normalise around 500 MW mean demand
    demand = (
        demand
        /
        demand.mean()
        *
        500.0
    )

    return demand


if DEMAND_CSV_PATH and os.path.isfile(DEMAND_CSV_PATH):
    _demand = pd.read_csv(DEMAND_CSV_PATH, parse_dates=["datetime"])
    if not {"datetime", "demand_MW"}.issubset(_demand.columns):
        raise ValueError("Demand CSV must contain datetime and demand_MW columns.")
    _demand["datetime"] = pd.to_datetime(_demand["datetime"], utc=True)
    _demand = _demand.set_index("datetime").sort_index()
    df = df.join(_demand[["demand_MW"]], how="inner")
    DEMAND_DATA_STATUS = "external CSV supplied; document source, units, and preprocessing"
else:
    df["demand_MW"] = create_demand_profile(df.index)
    DEMAND_DATA_STATUS = "synthetic constructed profile normalised to 500 MW mean; not empirical"
print("Demand data status:", DEMAND_DATA_STATUS)


# ================================================================
# 9. HYDRO PROFILE
# ================================================================
#
# Placeholder hydro availability profile.
#
# Replace this with actual reservoir / river-flow / hydro dispatch
# data in the final study.

month = df.index.month.values

hydro_monthly_cf = {
    1: 0.45,
    2: 0.42,
    3: 0.40,
    4: 0.38,
    5: 0.40,
    6: 0.55,
    7: 0.80,
    8: 0.85,
    9: 0.78,
    10: 0.65,
    11: 0.55,
    12: 0.48
}

df["hydro_cf"] = [
    hydro_monthly_cf[m]
    for m in month
]

df["hydro_MW"] = HYDRO_CAPACITY_MW * df["hydro_cf"]
if HYDRO_CSV_PATH and os.path.isfile(HYDRO_CSV_PATH):
    _hydro = pd.read_csv(HYDRO_CSV_PATH, parse_dates=["datetime"])
    if not {"datetime", "hydro_MW"}.issubset(_hydro.columns):
        raise ValueError("Hydro CSV must contain datetime and hydro_MW columns.")
    _hydro["datetime"] = pd.to_datetime(_hydro["datetime"], utc=True)
    _hydro = _hydro.set_index("datetime").sort_index()
    df = df.drop(columns=["hydro_MW"]).join(_hydro[["hydro_MW"]], how="inner")
    df["hydro_cf"] = (df["hydro_MW"] / HYDRO_CAPACITY_MW).clip(0, 1)
    HYDRO_DATA_STATUS = "external CSV supplied; document source, units, and preprocessing"
else:
    HYDRO_DATA_STATUS = "synthetic monthly capacity-factor placeholder; not observed hydropower"
print("Hydropower data status:", HYDRO_DATA_STATUS)
df.to_csv(os.path.join(PROCESSED_DIR, "model_timeseries_before_capacity_calibration.csv"))


# ================================================================
# 10. INITIAL RENEWABLE CAPACITY CALIBRATION
# ================================================================
#
# We calculate renewable capacities to approximately reach the
# target renewable-energy contribution.
#
# Solar and wind are calibrated using the observed resource profiles.

annual_load_MWh = (
    df["demand_MW"].sum()
    * RESAMPLE_HOURS
)

annual_hydro_MWh = (
    df["hydro_MW"].sum()
    * RESAMPLE_HOURS
)

target_RE_MWh = (
    TARGET_RENEWABLE_SHARE
    * annual_load_MWh
)

remaining_RE_MWh = max(
    target_RE_MWh - annual_hydro_MWh,
    0
)

solar_cf_mean = df["solar_cf"].mean()
wind_cf_mean = df["wind_cf"].mean()

# Allocate the remaining renewable energy between solar and wind.
solar_energy_target = remaining_RE_MWh * SOLAR_SHARE
wind_energy_target = remaining_RE_MWh * WIND_SHARE

solar_capacity = (
    solar_energy_target
    /
    (
        solar_cf_mean
        * len(df) * RESAMPLE_HOURS
    )
)

wind_capacity = (
    wind_energy_target
    /
    (
        wind_cf_mean
        * len(df) * RESAMPLE_HOURS
    )
)

solar_capacity = min(
    solar_capacity,
    MAX_SOLAR_MW
)

wind_capacity = min(
    wind_capacity,
    MAX_WIND_MW
)

df["solar_MW"] = (
    df["solar_cf"]
    * solar_capacity
)

df["wind_MW"] = (
    df["wind_cf"]
    * wind_capacity
)

df["renewable_available_MW"] = (
    df["solar_MW"]
    +
    df["wind_MW"]
    +
    df["hydro_MW"]
)

df["net_load_MW"] = (
    df["demand_MW"]
    -
    df["renewable_available_MW"]
)
df.to_csv(os.path.join(PROCESSED_DIR, "model_timeseries_calibrated.csv"))

print("\n===============================================")
print("SYSTEM CALIBRATION")
print("===============================================")

print(
    f"Solar capacity       : {solar_capacity:.2f} MW"
)

print(
    f"Wind capacity        : {wind_capacity:.2f} MW"
)

print(
    f"Hydro capacity       : {HYDRO_CAPACITY_MW:.2f} MW"
)

print(
    f"Annual load          : {annual_load_MWh:,.0f} MWh"
)

# Reviewer acceptance check: required aligned inputs, finite values, monotonic UTC index.
_required_model_columns = ["solar_cf", "wind_cf", "hydro_MW", "demand_MW", "renewable_available_MW"]
_missing_model_columns = [c for c in _required_model_columns if c not in df.columns]
if _missing_model_columns:
    raise ValueError(f"Model time series missing required columns: {_missing_model_columns}")
if df.empty:
    raise ValueError("Model time series is empty after alignment and missing-value handling.")
if not df.index.is_monotonic_increasing or df.index.has_duplicates:
    raise ValueError("Model time index must be unique and monotonically increasing.")
if not np.isfinite(df[_required_model_columns].to_numpy(dtype=float)).all():
    raise ValueError("Model inputs contain non-finite values after preprocessing.")
if (df["demand_MW"] < 0).any() or (df["hydro_MW"] < 0).any():
    raise ValueError("Demand and hydropower values must be non-negative.")

DASPI environment ready.

Coordinate validation successful.
Latitude  : 23.181500
Longitude : 79.986400
Location  : Jabalpur, Madhya Pradesh, India



/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Valid hourly observations: 8,760; expected approximately 8,760.
Missing counts before limited interpolation: {'solar_resource': 0, 'wind_speed_10m': 0, 'temperature_C': 0}
Missing counts after limited interpolation: {'solar_resource': 0, 'wind_speed_10m': 0, 'temperature_C': 0}


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



LOCATION RESOURCE DIAGNOSTIC
Mean solar resource indicator : 199.330
Mean wind speed               : 3.050 m/s
Maximum wind speed            : 10.420 m/s
Mean temperature              : 24.58 °C
Demand data status: synthetic constructed profile normalised to 500 MW mean; not empirical
Hydropower data status: synthetic monthly capacity-factor placeholder; not observed hydropower


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



SYSTEM CALIBRATION
Solar capacity       : 1127.25 MW
Wind capacity        : 1904.26 MW
Hydro capacity       : 100.00 MW
Annual load          : 4,380,000 MWh


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
def build_and_solve_DASPI(timeseries_df, techs, storage_params, spatial_config=None, fixed_energy_shares=None, grid_cost_override=None):
    """Multi-period MILP storage optimization formulation using scipy.optimize.milp."""
    import numpy as np
    import pandas as pd
    from scipy.optimize import milp, LinearConstraint, Bounds

    T = len(timeseries_df)
    dt = float(TIME_STEP)
    N = len(techs)

    # Number of variables per tech:
    # Power capacity: P_k (1)
    # Energy capacity: E_k (1)
    # Charging power: c_{k,t} (T)
    # Discharging power: d_{k,t} (T)
    # State of charge: soc_{k,t} (T)
    # Total per technology = 2 + 3*T

    # System-wide variables per timestep:
    # Unmet demand: unmet_t (T)
    # Grid import: grid_t (T)
    # Curtailment: curtail_t (T)

    vars_per_tech = 2 + 3 * T
    total_tech_vars = N * vars_per_tech

    # Variable indices mapping
    # For tech k:
    # P_k index: k * vars_per_tech
    # E_k index: k * vars_per_tech + 1
    # c_{k,t} index: k * vars_per_tech + 2 + t
    # d_{k,t} index: k * vars_per_tech + 2 + T + t
    # soc_{k,t} index: k * vars_per_tech + 2 + 2*T + t

    # System variables indices:
    # unmet_t: total_tech_vars + t
    # grid_t: total_tech_vars + T + t
    # curtail_t: total_tech_vars + 2*T + t

    total_vars = total_tech_vars + 3 * T

    # Objective function coefficients
    c_obj = np.zeros(total_vars)

    # Grid cost & unmet cost
    grid_cost = grid_cost_override if grid_cost_override is not None else 150.0
    unmet_cost = 2000.0

    for k, tech in enumerate(techs):
        p_capex = storage_params[tech]["power_capex"] * 1000  # to $/MW
        e_capex = storage_params[tech]["energy_capex"] * 1000  # to $/MWh

        # Optional spatial connection cost adjustment
        if spatial_config and "connection_cost_multiplier" in spatial_config:
            multiplier = spatial_config["connection_cost_multiplier"].get(tech, 1.0)
            p_capex *= multiplier

        if spatial_config and "distance_km" in spatial_config and "connection_cost_usd_per_mw_km" in spatial_config:
            dist = spatial_config["distance_km"].get(tech, 0.0)
            conn_unit_cost = spatial_config["connection_cost_usd_per_mw_km"]
            p_capex += dist * conn_unit_cost

        # Annualise
        ann_factor = 1.0 / MODEL_LIFETIME_YEARS

        # P_k and E_k objective coefficients
        c_obj[k * vars_per_tech] = p_capex * ann_factor
        c_obj[k * vars_per_tech + 1] = e_capex * ann_factor

        # Variable operational cost
        v_cost = storage_params[tech].get("variable_cost", 0.0) * 1000  # to $/MWh
        for t in range(T):
            c_obj[k * vars_per_tech + 2 + T + t] = v_cost * dt

    for t in range(T):
        c_obj[total_tech_vars + t] = unmet_cost * dt
        c_obj[total_tech_vars + T + t] = grid_cost * dt
        c_obj[total_tech_vars + 2*T + t] = 0.0

    # Bounds
    lower_bounds = np.zeros(total_vars)
    upper_bounds = np.full(total_vars, np.inf)

    for k, tech in enumerate(techs):
        # Max capacities
        max_p = MAX_STORAGE_POWER_MW
        max_e = MAX_STORAGE_ENERGY_MWH

        if spatial_config and "max_power_MW" in spatial_config:
            max_p = min(max_p, spatial_config["max_power_MW"].get(tech, max_p))
        if spatial_config and "max_energy_MWh" in spatial_config:
            max_e = min(max_e, spatial_config["max_energy_MWh"].get(tech, max_e))

        if spatial_config and "eligible" in spatial_config:
            if not spatial_config["eligible"].get(tech, True):
                max_p = 0.0
                max_e = 0.0

        upper_bounds[k * vars_per_tech] = max_p
        upper_bounds[k * vars_per_tech + 1] = max_e

        # Charging, discharging, and SOC bounds
        for t in range(T):
            # soc <= E_k
            pass

    for t in range(T):
        if spatial_config and "grid_import_limit_MW" in spatial_config:
            upper_bounds[total_tech_vars + T + t] = spatial_config["grid_import_limit_MW"]

    bounds = Bounds(lower_bounds, upper_bounds)

    # Constraints setup
    A_eq = []
    b_eq = []
    A_ineq = []
    b_ineq = []

    # 1. Supply-demand balance equation for each timestep:
    # Generation + discharge - charge + unmet + grid - curtail = demand
    for t in range(T):
        row = np.zeros(total_vars)
        row[total_tech_vars + t] = 1.0  # unmet_t
        row[total_tech_vars + T + t] = 1.0  # grid_t
        row[total_tech_vars + 2*T + t] = -1.0  # -curtail_t

        for k, tech in enumerate(techs):
            row[k * vars_per_tech + 2 + t] = -1.0  # -charge_{k,t}
            row[k * vars_per_tech + 2 + T + t] = 1.0  # +discharge_{k,t}

        A_eq.append(row)
        net_residual_demand = timeseries_df["demand_MW"].iloc[t] - timeseries_df["renewable_available_MW"].iloc[t]
        b_eq.append(net_residual_demand)

    # 2. Storage tracking constraints:
    # soc_t = soc_{t-1} * (1 - self_discharge) + dt * eta_ch * c_t - dt * d_t / eta_dis
    for k, tech in enumerate(techs):
        self_dis = storage_params[tech]["self_discharge"]
        eta_ch = storage_params[tech]["eta_ch"]
        eta_dis = storage_params[tech]["eta_dis"]

        for t in range(T):
            row = np.zeros(total_vars)
            row[k * vars_per_tech + 2 + 2*T + t] = 1.0  # soc_t
            row[k * vars_per_tech + 2 + t] = -dt * eta_ch
            row[k * vars_per_tech + 2 + T + t] = dt / eta_dis

            if t > 0:
                row[k * vars_per_tech + 2 + 2*T + (t-1)] = -(1.0 - self_dis)
                A_eq.append(row)
                b_eq.append(0.0)
            else:
                # Initial cycle SOC constraint
                row[k * vars_per_tech + 2 + 2*T + (T-1)] = -(1.0 - self_dis)
                A_eq.append(row)
                b_eq.append(0.0)

    # 3. Charging & discharging limits based on Power capacity:
    # c_t <= P_k and d_t <= P_k
    for k, tech in enumerate(techs):
        for t in range(T):
            row_ch = np.zeros(total_vars)
            row_ch[k * vars_per_tech] = -1.0  # -P_k
            row_ch[k * vars_per_tech + 2 + t] = 1.0  # c_{k,t}
            A_ineq.append(row_ch)
            b_ineq.append(0.0)

            row_dis = np.zeros(total_vars)
            row_dis[k * vars_per_tech] = -1.0  # -P_k
            row_dis[k * vars_per_tech + 2 + T + t] = 1.0  # d_{k,t}
            A_ineq.append(row_dis)
            b_ineq.append(0.0)

    # 4. Storage SOC limits based on Energy capacity:
    # soc_t <= E_k
    for k, tech in enumerate(techs):
        for t in range(T):
            row = np.zeros(total_vars)
            row[k * vars_per_tech + 1] = -1.0  # -E_k
            row[k * vars_per_tech + 2 + 2*T + t] = 1.0  # soc_{k,t}
            A_ineq.append(row)
            b_ineq.append(0.0)

    # 5. Technology min/max duration configuration constraint:
    # E_k >= P_k * min_duration and E_k <= P_k * max_duration
    for k, tech in enumerate(techs):
        min_dur = storage_params[tech]["min_duration"]
        max_dur = storage_params[tech]["max_duration"]

        row_min = np.zeros(total_vars)
        row_min[k * vars_per_tech] = min_dur
        row_min[k * vars_per_tech + 1] = -1.0
        A_ineq.append(row_min)
        b_ineq.append(0.0)

        row_max = np.zeros(total_vars)
        row_max[k * vars_per_tech] = -max_dur
        row_max[k * vars_per_tech + 1] = 1.0
        A_ineq.append(row_max)
        b_ineq.append(0.0)

    # 6. Fixed energy shares (for specific portfolio comparison runs)
    if fixed_energy_shares is not None:
        # Sum of E_k across all techs
        for k, tech in enumerate(techs):
            share = fixed_energy_shares.get(tech, 0.0)
            row = np.zeros(total_vars)
            row[k * vars_per_tech + 1] = 1.0
            for k2, tech2 in enumerate(techs):
                row[k2 * vars_per_tech + 1] -= share
            A_eq.append(row)
            b_eq.append(0.0)

    # Build linear constraint objects
    constraints = []
    if A_eq:
        A_eq = np.array(A_eq)
        constraints.append(LinearConstraint(A_eq, b_eq, b_eq))
    if A_ineq:
        A_ineq = np.array(A_ineq)
        constraints.append(LinearConstraint(A_ineq, -np.inf, b_ineq))

    integrality = np.zeros(total_vars)

    # Run MILP solver
    res = milp(c=c_obj, bounds=bounds, constraints=constraints, integrality=integrality)

    # Parse solution into Portfolio and Dispatch DataFrames
    portfolio_rows = []
    for k, tech in enumerate(techs):
        p_val = res.x[k * vars_per_tech] if res.success else 0.0
        e_val = res.x[k * vars_per_tech + 1] if res.success else 0.0
        p_capex = storage_params[tech]["power_capex"] * 1000
        e_capex = storage_params[tech]["energy_capex"] * 1000
        ann_cost = (p_val * p_capex + e_val * e_capex) / MODEL_LIFETIME_YEARS

        portfolio_rows.append({
            "Technology": tech,
            "Power_MW": p_val,
            "Energy_MWh": e_val,
            "Annualised_Capex_USD": ann_cost
        })

    portfolio_df = pd.DataFrame(portfolio_rows)

    dispatch_rows = []
    for t in range(T):
        unmet = res.x[total_tech_vars + t] if res.success else 0.0
        grid = res.x[total_tech_vars + T + t] if res.success else 0.0
        curtail = res.x[total_tech_vars + 2*T + t] if res.success else 0.0

        step_dict = {
            "datetime": timeseries_df.index[t],
            "demand_MW": timeseries_df["demand_MW"].iloc[t],
            "renewable_available_MW": timeseries_df["renewable_available_MW"].iloc[t],
            "unmet_MW": unmet,
            "grid_MW": grid,
            "curtailment_MW": curtail
        }
        for k, tech in enumerate(techs):
            step_dict[f"{tech}_charge_MW"] = res.x[k * vars_per_tech + 2 + t] if res.success else 0.0
            step_dict[f"{tech}_discharge_MW"] = res.x[k * vars_per_tech + 2 + T + t] if res.success else 0.0
            step_dict[f"{tech}_soc_MWh"] = res.x[k * vars_per_tech + 2 + 2*T + t] if res.success else 0.0

        dispatch_rows.append(step_dict)

    dispatch_df = pd.DataFrame(dispatch_rows).set_index("datetime")

    return res, portfolio_df, dispatch_df


In [3]:
# ================================================================
# SPATIAL ABLATION EXPERIMENT — SYNTHETIC DEMONSTRATION ONLY
# ================================================================
# This experiment tests whether spatial constraints are connected to the
# optimisation decision. It uses synthetic eligibility and connection-cost
# assumptions because validated candidate-site and interconnection data for
# Jabalpur have not been supplied. Do not report this as empirical geography.
# Run A: aggregate/coordinate-aware model without spatial restrictions.
# Run B: synthetic spatial eligibility restrictions, capacity bounds,
# distance-based connection-cost proxy, and an aggregate grid-import cap. Both runs use the same synthetic 250 USD/MWh grid-price stress assumption so the
# storage decision is sensitive enough to test the spatial mechanism. To keep
# the ablation tractable, it uses a fixed 168-step slice (28 days at 4-hour
# resolution), not the full-year headline case. This is a synthetic method test.

import os, json, numpy as np, pandas as pd

SPATIAL_OUT = os.path.join(OUTPUT_DIR, "results", "spatial_ablation")
os.makedirs(SPATIAL_OUT, exist_ok=True)

# Deliberately explicit demonstration assumptions, to be replaced with
# sourced site-screening and interconnection data before a real-world claim.
SPATIAL_DEMO_CONFIG = {
    "eligible": {
        "Li-ion": True,
        "Flow Battery": True,
        "Pumped Hydro": False,
        "Hydrogen": False,
        "Thermal Storage": True
    },
    "connection_cost_multiplier": {
        "Li-ion": 1.15,
        "Flow Battery": 1.20,
        "Pumped Hydro": 1.00,
        "Hydrogen": 1.00,
        "Thermal Storage": 1.10
    },
    "distance_km": {
        "Li-ion": 5.0,
        "Flow Battery": 12.0,
        "Pumped Hydro": 35.0,
        "Hydrogen": 20.0,
        "Thermal Storage": 8.0
    },
    "connection_cost_usd_per_mw_km": 1000.0,
    "max_power_MW": {
        "Li-ion": 1000.0,
        "Flow Battery": 800.0,
        "Pumped Hydro": 0.0,
        "Hydrogen": 0.0,
        "Thermal Storage": 500.0
    },
    "max_energy_MWh": {
        "Li-ion": 8000.0,
        "Flow Battery": 12000.0,
        "Pumped Hydro": 0.0,
        "Hydrogen": 0.0,
        "Thermal Storage": 6000.0
    },
    "grid_import_limit_MW": 350.0,
    "assumption_status": "synthetic demonstration; not observed Jabalpur data",
    "eligible_site_logic": "technology-level hard eligibility and capacity proxy, not mapped candidate-site GIS",
    "connection_cost_logic": "synthetic distance_km x USD/MW-km plus equivalent capex multiplier; not a sourced tariff",
    "grid_connection_logic": "synthetic aggregate grid-import limit; not a network topology or power-flow model",
    "grid_cost_override": 250.0,
    "grid_cost_status": "synthetic 250 USD/MWh stress assumption applied identically to A and B; not a sourced tariff",
    "temporal_window": "first 168 model steps (28 days at 4-hour resolution); not full-year"
}

# Use the first 168 model steps (28 days at 4-hour resolution) so two
# optimisation runs are computationally tractable. Both runs use the exact
# same time slice and grid-cost stress assumption.
if "df" not in globals() or not isinstance(df, pd.DataFrame):
    raise RuntimeError("Run the main model/resource-processing cell successfully before spatial ablation.")
spatial_demo_df = df.iloc[:min(168, len(df))].copy()
if len(spatial_demo_df) < 24:
    raise ValueError("Spatial ablation requires at least 24 model time steps.")
print(f"Spatial ablation window: {len(spatial_demo_df)} steps x {RESAMPLE_HOURS} h = {len(spatial_demo_df)*RESAMPLE_HOURS} hours")

# The baseline solve uses the same inputs with no spatial restrictions.
print("\nRunning spatial ablation A: no spatial constraints")
spatial_A_result, spatial_A_portfolio, spatial_A_dispatch = build_and_solve_DASPI(
    spatial_demo_df, TECHNOLOGIES, STORAGE, grid_cost_override=SPATIAL_DEMO_CONFIG["grid_cost_override"]
)

print("\nRunning spatial ablation B: synthetic eligibility and connection-cost proxy")
spatial_B_result, spatial_B_portfolio, spatial_B_dispatch = build_and_solve_DASPI(
    spatial_demo_df, TECHNOLOGIES, STORAGE, spatial_config=SPATIAL_DEMO_CONFIG,
    grid_cost_override=SPATIAL_DEMO_CONFIG["grid_cost_override"]
)

def _spatial_summary(label, result, portfolio, dispatch):
    dt_hours = float(RESAMPLE_HOURS)
    unmet = float(dispatch["unmet_MW"].sum() * dt_hours)
    grid = float(dispatch["grid_MW"].sum() * dt_hours)
    curtail = float(dispatch["curtailment_MW"].sum() * dt_hours)
    return {
        "Run": label,
        "Solver_Success": bool(result.success),
        "Solver_Status": str(result.message),
        "Objective_Value_model_units": float(result.fun),
        "Storage_Power_MW": float(portfolio["Power_MW"].sum()),
        "Storage_Energy_MWh": float(portfolio["Energy_MWh"].sum()),
        "Grid_Imports_MWh": grid,
        "Unserved_Energy_MWh": unmet,
        "Curtailment_MWh": curtail,
        "Interpretation": "synthetic spatial ablation; not validated regional performance"
    }

spatial_ablation_summary = pd.DataFrame([
    _spatial_summary("A_no_spatial_constraints", spatial_A_result, spatial_A_portfolio, spatial_A_dispatch),
    _spatial_summary("B_synthetic_spatial_constraints", spatial_B_result, spatial_B_portfolio, spatial_B_dispatch)
])
spatial_portfolio = pd.concat([
    spatial_A_portfolio.assign(Run="A_no_spatial_constraints"),
    spatial_B_portfolio.assign(Run="B_synthetic_spatial_constraints")
], ignore_index=True)
spatial_ablation_summary.to_csv(os.path.join(SPATIAL_OUT, "spatial_ablation_summary.csv"), index=False)
spatial_portfolio.to_csv(os.path.join(SPATIAL_OUT, "spatial_ablation_portfolios.csv"), index=False)
with open(os.path.join(SPATIAL_OUT, "spatial_demo_assumptions.json"), "w", encoding="utf-8") as f:
    json.dump(SPATIAL_DEMO_CONFIG, f, indent=2)
spatial_input_hash = hashlib.sha256(spatial_demo_df.to_csv().encode("utf-8")).hexdigest()
with open(os.path.join(SPATIAL_OUT, "spatial_run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump({
        "run_time_utc": datetime.now(timezone.utc).isoformat(),
        "input_sha256": spatial_input_hash,
        "n_steps": int(len(spatial_demo_df)),
        "timestep_hours": float(RESAMPLE_HOURS),
        "modeled_hours": float(len(spatial_demo_df) * RESAMPLE_HOURS),
        "paired_input_same_for_A_and_B": True,
        "spatial_evidence_status": "synthetic method test only; not empirical GIS/network data",
        "solver_A_success": bool(spatial_A_result.success),
        "solver_B_success": bool(spatial_B_result.success)
    }, f, indent=2)

print("\nSPATIAL ABLATION SUMMARY")
print(spatial_ablation_summary.to_string(index=False))
print("\nPortfolio comparison")
print(spatial_portfolio.to_string(index=False))
print("\nSaved spatial ablation outputs to:", SPATIAL_OUT)
print("NOTE: this tests synthetic eligibility/capacity/distance-cost/grid-import constraints only. It is NOT a network-topology or power-flow model. Replace assumptions with traceable GIS/network data before making a Jabalpur-specific claim.")


Spatial ablation window: 168 steps x 4 h = 672 hours

Running spatial ablation A: no spatial constraints


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Running spatial ablation B: synthetic eligibility and connection-cost proxy


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



SPATIAL ABLATION SUMMARY
                            Run  Solver_Success                                                   Solver_Status  Objective_Value_model_units  Storage_Power_MW  Storage_Energy_MWh  Grid_Imports_MWh  Unserved_Energy_MWh  Curtailment_MWh                                                 Interpretation
       A_no_spatial_constraints            True Optimization terminated successfully. (HiGHS Status 7: Optimal)                 4.286054e+07          0.000000            0.000000     171442.161899             0.000000     31616.170303 synthetic spatial ablation; not validated regional performance
B_synthetic_spatial_constraints            True Optimization terminated successfully. (HiGHS Status 7: Optimal)                 5.090255e+07        178.535821         1751.291159     167811.827598           228.192912     11792.816545 synthetic spatial ablation; not validated regional performance

Portfolio comparison
     Technology   Power_MW  Energy_MWh  Annualised_Capex_U

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Reviewer experiment harness and evidence boundary

Run this cell after the main model and spatial-ablation cell complete. It independently optimises No Storage, Li-ion Only, Hydrogen-only LDES, Fixed Hybrid, and DASPI-C on the same time series and saves dispatch traces and a summary table. The default is the full available horizon; set `BASELINE_MAX_STEPS = 168` only for a debugging smoke test, never as an annual result.

The code includes a synthetic spatial demonstration with hard technology eligibility/capacity limits, distance-based connection-cost proxies, and an aggregate grid-import limit. It does **not** implement GIS candidate-site allocation or full transmission topology/power-flow constraints. Those require real, sourced site/network inputs and further model development.

If empirical demand/hydropower CSVs are unavailable, the code retains explicitly labelled constructed profiles. Code changes alone cannot convert these into empirical Jabalpur observations.

In [ ]:
# ================================================================
# REVIEWER EXPERIMENT HARNESS — MATCHED INDEPENDENT MILP BASELINES
# ================================================================
# Run after the main model/resource-processing cell and the spatial cell.
# Full-horizon mode is the publication run. A short run is debugging only.
# A successful execution is not, by itself, empirical validation.

import os, json, time, hashlib, platform, sys, re
from datetime import datetime, timezone

RUN_REVIEWER_BASELINES = True
BASELINE_MAX_STEPS = None  # None = full available horizon; 168 = smoke test only
BASELINE_OUT = os.path.join(RESULTS_DIR, "independent_baselines")
os.makedirs(BASELINE_OUT, exist_ok=True)


def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _event_table_from_unserved(dispatch, timestep_hours, threshold_mw=1e-9):
    """Create residual-unserved-energy events from a scenario dispatch trace."""
    if dispatch is None or "unmet_MW" not in dispatch.columns:
        return pd.DataFrame(columns=["event_id", "start_index", "end_index", "duration_h", "peak_deficit_MW", "energy_deficit_MWh"])
    x = np.asarray(dispatch["unmet_MW"], dtype=float)
    mask = x > threshold_mw
    starts = np.where(mask & ~np.r_[False, mask[:-1]])[0]
    ends = np.where(mask & ~np.r_[mask[1:], False])[0]
    rows = []
    for eid, (a, b) in enumerate(zip(starts, ends), start=1):
        seg = x[a:b+1]
        rows.append({
            "event_id": eid, "start_index": int(a), "end_index": int(b),
            "duration_h": float(len(seg) * timestep_hours),
            "peak_deficit_MW": float(seg.max()),
            "energy_deficit_MWh": float(seg.sum() * timestep_hours)
        })
    return pd.DataFrame(rows, columns=["event_id", "start_index", "end_index", "duration_h", "peak_deficit_MW", "energy_deficit_MWh"])


def _safe_total(frame, col, dt):
    return float(frame[col].sum() * dt) if frame is not None and col in frame.columns else np.nan


if RUN_REVIEWER_BASELINES:
    if "df" not in globals():
        raise RuntimeError("Main model DataFrame 'df' is missing. Run the main resource-processing cell successfully first.")
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise ValueError("'df' must be a non-empty pandas DataFrame.")
    _required = ["renewable_available_MW", "demand_MW"]
    _missing = [c for c in _required if c not in df.columns]
    if _missing:
        raise ValueError(f"df is missing required baseline columns {_missing}; available={list(df.columns)}")
    if not np.isfinite(df[_required].to_numpy(dtype=float)).all():
        raise ValueError("Baseline inputs contain non-finite renewable or demand values.")

    baseline_data = df.copy()
    if BASELINE_MAX_STEPS is not None:
        if int(BASELINE_MAX_STEPS) < 2:
            raise ValueError("BASELINE_MAX_STEPS must be >= 2 or None.")
        baseline_data = baseline_data.iloc[:int(BASELINE_MAX_STEPS)].copy()
    if len(baseline_data) < 2:
        raise ValueError("Baseline experiment needs at least two time steps.")

    baseline_cases = [
        ("No Storage", [], {}, None),
        ("Li-ion Only", ["Li-ion"], {"Li-ion": STORAGE["Li-ion"]}, None),
        ("Single LDES (Hydrogen-only)", ["Hydrogen"], {"Hydrogen": STORAGE["Hydrogen"]}, None),
        ("Fixed Hybrid (Li-ion/Hydrogen 50:50 energy)", ["Li-ion", "Hydrogen"],
         {"Li-ion": STORAGE["Li-ion"], "Hydrogen": STORAGE["Hydrogen"]},
         {"Li-ion": 0.5, "Hydrogen": 0.5}),
        ("DASPI-C", TECHNOLOGIES, STORAGE, None)
    ]

    baseline_rows, baseline_portfolios = [], []
    dispatches = {}
    baseline_run_metadata = {
        "run_started_utc": datetime.now(timezone.utc).isoformat(),
        "location_name": LOCATION_NAME, "latitude": LATITUDE, "longitude": LONGITUDE,
        "start_date_config": START_DATE, "end_date_config": END_DATE,
        "time_step_hours": float(RESAMPLE_HOURS), "n_steps": int(len(baseline_data)),
        "modeled_hours": float(len(baseline_data) * RESAMPLE_HOURS),
        "baseline_max_steps": BASELINE_MAX_STEPS,
        "demand_data_status": DEMAND_DATA_STATUS, "hydro_data_status": HYDRO_DATA_STATUS,
        "model_lifetime_years": MODEL_LIFETIME_YEARS,
        "grid_cost_usd_per_mwh": 150.0,
        "grid_import_policy": "same model grid variable/cost/bounds across cases; inspect solver formulation before interpreting",
        "solver": "scipy.optimize.milp (HiGHS backend)",
        "python_version": sys.version, "platform": platform.platform(),
        "numpy_version": np.__version__, "pandas_version": pd.__version__,
        "scipy_version": __import__("scipy").__version__,
        "input_sha256": hashlib.sha256(baseline_data.to_csv().encode("utf-8")).hexdigest(),
        "interpretation_boundary": "scenario comparison under the same model inputs; not empirical regional validation"
    }

    for case_name, case_techs, case_params, fixed_shares in baseline_cases:
        print("\n" + "="*72 + "\nINDEPENDENT OPTIMISATION: " + case_name + "\n" + "="*72)
        started = time.time()
        case_file = re.sub(r"[^a-z0-9]+", "_", case_name.lower()).strip("_")
        try:
            solved, case_portfolio, case_dispatch = build_and_solve_DASPI(
                baseline_data, case_techs, case_params,
                fixed_energy_shares=fixed_shares
            )
            runtime_seconds = time.time() - started
            dt_h = float(RESAMPLE_HOURS)
            load_mwh = _safe_total(case_dispatch, "demand_MW", dt_h)
            unserved_mwh = _safe_total(case_dispatch, "unmet_MW", dt_h)
            grid_mwh = _safe_total(case_dispatch, "grid_MW", dt_h)
            curtail_mwh = _safe_total(case_dispatch, "curtailment_MW", dt_h)
            renewable_mwh = _safe_total(case_dispatch, "renewable_available_MW", dt_h)
            annualized_cost_usd = float(case_portfolio["Annualised_Capex_USD"].sum()) if len(case_portfolio) and "Annualised_Capex_USD" in case_portfolio.columns else 0.0
            reliability_pct = 100.0 * (1.0 - unserved_mwh / load_mwh) if load_mwh > 0 else np.nan
            dispatches[case_name] = case_dispatch.copy()
            case_dispatch.to_csv(os.path.join(BASELINE_OUT, f"{case_file}_dispatch.csv"), index=True)
            # Scenario-specific residual unserved events are archived separately from the pre-storage renewable-deficit events.
            residual_events = _event_table_from_unserved(case_dispatch, dt_h)
            residual_events.to_csv(os.path.join(BASELINE_OUT, f"{case_file}_residual_unserved_event_table.csv"), index=False)
            case_tscr = calculate_TSCR_event_screen(events, case_portfolio, case_params) if len(case_techs) else 0.0
            row = {
                "Scenario": case_name, "Solver_Status": str(solved.message), "Solver_Success": bool(solved.success),
                "Objective_Value_model_units": float(solved.fun),
                "Objective_Interpretation": "model objective; inspect objective components/scaling before USD-equivalent economic claims",
                "Runtime_seconds": runtime_seconds,
                "Power_MW": float(case_portfolio["Power_MW"].sum()) if len(case_portfolio) else 0.0,
                "Energy_MWh": float(case_portfolio["Energy_MWh"].sum()) if len(case_portfolio) else 0.0,
                "Annualised_Storage_CAPEX_USD": annualized_cost_usd,
                "Grid_Imports_MWh": grid_mwh, "Unserved_Energy_MWh": unserved_mwh,
                "Curtailment_MWh": curtail_mwh, "Renewable_Available_MWh": renewable_mwh,
                "Load_MWh": load_mwh, "Reliability_percent_grid_supported": reliability_pct,
                "TSCR_event_screen_fraction": case_tscr,
                "Residual_Unserved_Event_Count": int(len(residual_events)),
                "Interpretation": "independently solved scenario on shared modeled horizon; not empirical regional validation"
            }
            if len(case_portfolio):
                _p = case_portfolio.copy(); _p.insert(0, "Scenario", case_name); baseline_portfolios.append(_p)
            baseline_rows.append(row)
        except Exception as exc:
            runtime_seconds = time.time() - started
            baseline_rows.append({"Scenario": case_name, "Solver_Status": "FAILED: " + repr(exc),
                                  "Solver_Success": False, "Runtime_seconds": runtime_seconds,
                                  "Interpretation": "failed run; no numerical performance comparison permitted"})
            print("CASE FAILED:", repr(exc))

    # Compute RASV only after the No Storage reference dispatch has completed.
    no_storage_dispatch = dispatches.get("No Storage")
    rows_by_name = {r["Scenario"]: r for r in baseline_rows}
    for case_name, _, _, _ in baseline_cases:
        if case_name == "No Storage" or case_name not in dispatches:
            continue
        # Recover matching portfolio from the portfolio output rows; if unavailable RASV remains undefined.
        pframes = [p for p in baseline_portfolios if len(p) and p["Scenario"].iloc[0] == case_name]
        pframe = pframes[0].drop(columns=["Scenario"]) if pframes else pd.DataFrame()
        r = calculate_RASV(no_storage_dispatch, dispatches[case_name], pframe, float(RESAMPLE_HOURS))
        rows_by_name[case_name].update(r)

    baseline_results_df = pd.DataFrame(baseline_rows)
    baseline_results_df.to_csv(os.path.join(BASELINE_OUT, "independent_baseline_summary.csv"), index=False)
    if baseline_portfolios:
        pd.concat(baseline_portfolios, ignore_index=True).to_csv(os.path.join(BASELINE_OUT, "independent_baseline_portfolios.csv"), index=False)

    # Archive the common pre-storage renewable-deficit events and the DMI arithmetic check.
    events.to_csv(os.path.join(RESULTS_DIR, "pre_storage_renewable_deficit_event_table.csv"), index=False)
    dmi_audit = {
        "formula": "sum(E_k * max(d_k - reference_duration_h, 0)) / (reference_duration_h * sum(E_k))",
        "reference_duration_hours": float(REFERENCE_DURATION_HOURS),
        "units": "dimensionless; not bounded to [0,1]",
        "DMI_reported_for_pre_storage_events": float(DMI) if np.isfinite(DMI) else None,
        "DMI_recalculated_from_archived_events": float(calculate_DMI(pd.read_csv(os.path.join(RESULTS_DIR, "pre_storage_renewable_deficit_event_table.csv")))) if len(events) else None,
        "DMI_match": bool(np.isclose(DMI, calculate_DMI(events), equal_nan=True)),
        "event_table_sha256": _sha256_file(os.path.join(RESULTS_DIR, "pre_storage_renewable_deficit_event_table.csv")),
        "scope": "arithmetic consistency check on the saved event table; not independent validation of raw NASA/input generation"
    }
    with open(os.path.join(RESULTS_DIR, "dmi_recalculation_audit.json"), "w", encoding="utf-8") as f: json.dump(dmi_audit, f, indent=2)

    baseline_run_metadata["run_finished_utc"] = datetime.now(timezone.utc).isoformat()
    baseline_run_metadata["all_cases_successful"] = bool(len(baseline_results_df) == len(baseline_cases) and baseline_results_df["Solver_Success"].fillna(False).all())
    baseline_run_metadata["baseline_results_sha256"] = _sha256_file(os.path.join(BASELINE_OUT, "independent_baseline_summary.csv"))
    with open(os.path.join(BASELINE_OUT, "run_metadata.json"), "w", encoding="utf-8") as f: json.dump(baseline_run_metadata, f, indent=2)

    # Run configuration + provenance manifest: include generated input and result files.
    provenance = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "location": LOCATION_NAME, "coordinates": {"latitude": LATITUDE, "longitude": LONGITUDE},
        "date_range": {"start": START_DATE, "end": END_DATE, "time_standard": "UTC"},
        "NASA_POWER_parameters": ["ALLSKY_SFC_SW_DWN", "WS10M", "T2M"],
        "solar_conversion": {"input_unit_assumption": SOLAR_INPUT_UNIT_ASSUMPTION, "derate": PV_SYSTEM_DERATE,
                             "temperature_coefficient_per_C": PV_TEMPERATURE_COEFFICIENT_PER_C,
                             "reference_temperature_C": PV_REFERENCE_TEMPERATURE_C,
                             "clip_rule": "irradiance clipped at zero; temperature factor clipped to [0,1.2]; solar_cf clipped to [0,1]"},
        "wind_conversion": {"measurement_height_m": WIND_MEASUREMENT_HEIGHT_M, "hub_height_m": WIND_HUB_HEIGHT_M,
                            "shear_exponent": WIND_SHEAR_EXPONENT, "curve": WIND_TURBINE_MODEL,
                            "cut_in_m_s": 3.0, "rated_m_s": 12.0, "cut_out_m_s": 25.0,
                            "clip_rule": "power coefficient clipped to [0,1]"},
        "demand_status": DEMAND_DATA_STATUS, "hydro_status": HYDRO_DATA_STATUS,
        "TSCR_definition": TSCR_DEFINITION, "RASV_definition": RASV_DEFINITION,
        "DMI_audit": dmi_audit,
        "baseline_all_successful": baseline_run_metadata["all_cases_successful"],
        "warning": "A run is not empirical validation; demand/hydro/cost assumptions and synthetic spatial constraints must be disclosed."
    }
    with open(os.path.join(RESULTS_DIR, "run_provenance.json"), "w", encoding="utf-8") as f: json.dump(provenance, f, indent=2)

    # SHA-256 manifest created last; omit manifest itself from its own contents.
    manifest_rows = []
    for root, _, files in os.walk(OUTPUT_DIR):
        for filename in sorted(files):
            path = os.path.join(root, filename)
            if filename == "sha256_manifest.csv": continue
            try:
                manifest_rows.append({"relative_path": os.path.relpath(path, OUTPUT_DIR), "size_bytes": os.path.getsize(path), "sha256": _sha256_file(path)})
            except OSError:
                pass
    pd.DataFrame(manifest_rows).to_csv(os.path.join(RESULTS_DIR, "sha256_manifest.csv"), index=False)

    print("\nINDEPENDENT BASELINE RESULTS")
    print(baseline_results_df.to_string(index=False))
    print("\nDMI audit:", json.dumps(dmi_audit, indent=2))
    print("\nSaved baseline dispatches, scenario residual-event tables, metric audit, provenance and hashes to:", OUTPUT_DIR)
    if not baseline_run_metadata["all_cases_successful"]:
        print("WARNING: one or more cases failed; no comparative superiority claim is permitted.")
    if BASELINE_MAX_STEPS is not None:
        print("WARNING: short-horizon smoke-test results are not annual results and must not be reported as publication findings.")
else:
    print("Baseline runs skipped. Set RUN_REVIEWER_BASELINES=True to execute.")


--- Logging error ---
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
  File "/usr/local/lib/python3.13/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
    ~~~~~~~~~~~~~~~~~^
        self.pub_thread,
        ^^^^^^^^^^^^^^^^
    ...<3 lines>...
        ident=self.topic,
        ^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py", line 820, in send
    msg = self.msg(
        msg_or_type,
    ...<3 lines>...
        metadata=metadata,
    )
  File "/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py", line 661, in msg
    header = self.msg_header(msg_type) if header is None else header
             ~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py", line 644, in msg_header
    return msg_header(self.msg_id, msg_type, self.username, self.s


INDEPENDENT OPTIMISATION: Li-ion Only


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


CASE FAILED: NameError("name 'calculate_TSCR_event_screen' is not defined")

INDEPENDENT OPTIMISATION: Single LDES (Hydrogen-only)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


CASE FAILED: NameError("name 'calculate_TSCR_event_screen' is not defined")

INDEPENDENT OPTIMISATION: Fixed Hybrid (Li-ion/Hydrogen 50:50 energy)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


CASE FAILED: NameError("name 'calculate_TSCR_event_screen' is not defined")

INDEPENDENT OPTIMISATION: DASPI-C


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Reviewer acceptance gate
Run after the main model, spatial ablation, and baseline harness. This final cell writes an explicit criterion/status register.


In [ ]:
# ================================================================
# REVIEWER ACCEPTANCE GATE — RUN AFTER ALL EXPERIMENT CELLS
# ================================================================
# This gate does not fabricate missing empirical evidence. It reports which
# reviewer requirements are code-implemented versus data/experiment-dependent.

acceptance_rows = []
def _gate(item, status, evidence, blocker=""):
    acceptance_rows.append({"Reviewer_Criterion": item, "Code_Status": status, "Evidence_or_Output": evidence, "Remaining_Blocker": blocker})

_gate("DMI definition", "IMPLEMENTED + arithmetic checked", "metric_definitions/run DMI audit and pre_storage_renewable_deficit_event_table.csv", "A clean independent rerun and raw input archive are still needed for full reproduction.")
_gate("Independent baselines", "IMPLEMENTED; execution status must be inspected", "results/independent_baselines/independent_baseline_summary.csv", "Every case must report Solver_Success=True; failures/infeasibility must remain visible.")
_gate("Spatial optimisation constraints", "SYNTHETIC mechanism test only", "results/spatial_ablation/spatial_ablation_summary.csv and spatial_run_metadata.json", "Replace synthetic eligibility, distance and grid cap with sourced GIS/site/network/interconnection data for regional claims.")
_gate("Empirical demand/hydro/cost inputs", "INPUT-PROVENANCE GATE", "run_provenance.json plus demand/hydro status fields", "Supply source-backed demand and hydro CSVs and cited/versioned cost parameters; default profiles are illustrative.")
_gate("Resource-to-power conversion", "PARAMETERS ARCHIVED; simplified model", "run_provenance.json and NASA request metadata/raw JSON", "Verify NASA response units and conversion assumptions; generic wind curve is not a manufacturer curve or validated PV model.")
_gate("TSCR definition/implementation", "RECONCILED AS OPTIMISTIC EVENT SCREEN", "TSCR_event_screen_fraction in baseline summary; definition in run_provenance.json", "Not chronological dispatch reliability; never call it the fraction actually served unless dispatch logic is changed and validated.")
_gate("RASV definition/implementation", "RECONCILED WITH MATCHED NO-STORAGE REFERENCE", "RASV_MWh_per_MUSD and RASV_status in baseline summary", "Only interpret when No Storage and case runs succeed with same horizon and positive annualised CAPEX.")

acceptance_df = pd.DataFrame(acceptance_rows)
acceptance_df.to_csv(os.path.join(RESULTS_DIR, "reviewer_acceptance_gate.csv"), index=False)
print(acceptance_df.to_string(index=False))
print("\nIMPORTANT: this checklist distinguishes code implementation from empirical validation. A code path or synthetic test does not prove a regional result.")
